In [28]:
import crypten
import torch

In [50]:
# def backward(ctx, grad_output):
def backward(
    input, kernel_size, padding=0, stride=None, dilation=1, ceil_mode=False, grad_output=None, is_print=False
):
    # input, kernel_size, padding, stride, dilation, ceil_mode = ctx.saved_tensors
    print(input.size())
    batches, channels, height, width = input.size()
    
    input.share = torch.nn.functional.unfold(
        input.share.float(), kernel_size=kernel_size, dilation=dilation, padding=padding, stride=stride
    ) # (batch_size,channel_size*kernel_h*kernel_w,num_windows)
    input.share = input.share.long()
    unfolded_input = input.view(batches,channels,kernel_size**2,-1) # (batch_size,channel_size,kernel_h*kernel_w,num_windows)
    if is_print:
        print("unfolded_input：",unfolded_input.get_plain_text())
        print("the shape of unfolded_input：",unfolded_input.shape)
    
    e_unfolded_input = unfolded_input.exp()
    unfolded_input_sum = e_unfolded_input.sum(dim=2)
    if is_print:
        print("the sum of unfolded_input：",unfolded_input_sum.get_plain_text())
    mask_sum_max = torch.clamp(unfolded_input_sum.get_plain_text(),float('1e-2'), float('1e8')) # limit
    if is_print:
        print("the limited sum of unfolded_input",mask_sum_max)
    
    input_weight = e_unfolded_input / mask_sum_max.unsqueeze(2)
    input_weight = input_weight.get_plain_text()
    if is_print:
        print("input_weight:",input_weight)
        print("the shape of input_weight:",input_weight.shape)
    grad_output = grad_output.view(batches,channels,1,-1)
    if is_print:
        print("grad_output:",grad_output.get_plain_text())
        print("the shape of grad_output:",grad_output.shape)
    
    grad_input = grad_output*input_weight
    if is_print:
        print("grad_input:",grad_input.get_plain_text())
        print("the shape of grad_input:",grad_input.shape)
    grad_input = grad_input.view(batches,channels*kernel_size**2,-1)
    if is_print:
        print("grad_input:",grad_input.get_plain_text())
        print("the shape of grad_input:",grad_input.shape)
    grad_input.share = torch.nn.functional.fold(
        grad_input.share.float(), (height, width), kernel_size=kernel_size, dilation=dilation, padding=padding, stride=stride
    )
    grad_input.share = grad_input.share.long()
    if is_print:
        print("grad_input:",grad_input.get_plain_text())
        print("the shape of grad_input:",grad_input.shape)
    
    return grad_input

In [59]:
crypten.init()

# 定义输入数据和输出梯度
data_input = torch.randn(1, 1, 3, 3)
input = crypten.cryptensor(data_input)
print(input.get_plain_text())
print(input.shape)

diff_output = torch.randn(1, 1, 2, 2)
grad_output = crypten.cryptensor(diff_output)
print(grad_output.get_plain_text())
print(grad_output.shape)

tensor([[[[ 0.5509, -0.2365, -0.5658],
          [-0.1985, -0.7023,  1.3402],
          [ 1.3128, -0.4129, -0.2780]]]])
torch.Size([1, 1, 3, 3])
tensor([[[[ 1.1792,  0.8500],
          [ 0.9002, -1.4720]]]])
torch.Size([1, 1, 2, 2])


In [60]:
batches, channels, height, width = data_input.size()
kernel_size, stride = 2, 1

unfolded_input = torch.nn.functional.unfold(data_input, kernel_size=kernel_size, stride=stride) # (batch_size,channel_size*kernel_h*kernel_w,num_windows)
unfolded_input = unfolded_input.view(batches,channels,kernel_size**2,-1) # (batch_size,channel_size,kernel_h*kernel_w,num_windows)
print("unfolded_input：",unfolded_input)
print("the shape of unfolded_input：",unfolded_input.shape) 

unfolded_input_sum = torch.exp(unfolded_input).sum(dim=2)
print("the sum of unfolded_input：",unfolded_input_sum)
mask_sum_max = torch.clamp(unfolded_input_sum, float('1e-2'), float('1e8')) # limit
print("the limited unfolded_input：",mask_sum_max)

diff_input = torch.exp(unfolded_input) / mask_sum_max.unsqueeze(2)
print("input_weight：",diff_input)
print("the weight of input_weight：",diff_input.shape)
diff_output = diff_output.view(batches,channels,1,-1)
print("grad_output：",diff_output)
print("the shape of grad_putput：",diff_output.shape)

diff_input = (diff_output*diff_input) 
print("grad_input：",diff_input)
print("the shape of grad_input：",diff_input.shape)
diff_input = diff_input.view(batches,channels*kernel_size**2,-1)
print("grad_input：",diff_input)
print("the shape of grad_input：",diff_input.shape)
diff_input = torch.nn.functional.fold(diff_input, (height, width), kernel_size=kernel_size, stride=stride)
print("grad_input：",diff_input)
print("the shape of grad_input：",diff_input.shape)

unfolded_input： tensor([[[[ 0.5509, -0.2366, -0.1985, -0.7023],
          [-0.2366, -0.5658, -0.7023,  1.3402],
          [-0.1985, -0.7023,  1.3128, -0.4129],
          [-0.7023,  1.3402, -0.4129, -0.2780]]]])
the shape of unfolded_input： torch.Size([1, 1, 4, 4])
the sum of unfolded_input： tensor([[[3.8396, 5.6725, 5.6939, 5.7343]]])
the limited unfolded_input： tensor([[[3.8396, 5.6725, 5.6939, 5.7343]]])
input_weight： tensor([[[[0.4518, 0.1392, 0.1440, 0.0864],
          [0.2056, 0.1001, 0.0870, 0.6661],
          [0.2136, 0.0873, 0.6528, 0.1154],
          [0.1290, 0.6734, 0.1162, 0.1321]]]])
the weight of input_weight： torch.Size([1, 1, 4, 4])
grad_output： tensor([[[[ 1.1792,  0.8501,  0.9002, -1.4720]]]])
the shape of grad_putput： torch.Size([1, 1, 1, 4])
grad_input： tensor([[[[ 0.5328,  0.1183,  0.1296, -0.1272],
          [ 0.2424,  0.0851,  0.0783, -0.9805],
          [ 0.2518,  0.0742,  0.5876, -0.1699],
          [ 0.1522,  0.5724,  0.1046, -0.1944]]]])
the shape of grad_inpu

In [61]:
grad_input = backward(
    input, kernel_size=2, padding=0, stride=1, dilation=1, ceil_mode=False, grad_output=grad_output, is_print=True
)

torch.Size([1, 1, 3, 3])
unfolded_input： tensor([[[[ 0.5509, -0.2365, -0.1985, -0.7023],
          [-0.2365, -0.5658, -0.7023,  1.3402],
          [-0.1985, -0.7023,  1.3128, -0.4129],
          [-0.7023,  1.3402, -0.4129, -0.2780]]]])
the shape of unfolded_input： torch.Size([1, 1, 4, 4])
the sum of unfolded_input： tensor([[[3.8382, 5.6513, 5.6715, 5.7116]]])
the limited sum of unfolded_input tensor([[[3.8382, 5.6513, 5.6715, 5.7116]]])
input_weight: tensor([[[[0.4510, 0.1398, 0.1449, 0.0867],
          [0.2059, 0.1006, 0.0873, 0.6648],
          [0.2141, 0.0876, 0.6510, 0.1159],
          [0.1290, 0.6719, 0.1168, 0.1325]]]])
the shape of input_weight: torch.Size([1, 1, 4, 4])
grad_output: tensor([[[[ 1.1792,  0.8500,  0.9002, -1.4720]]]])
the shape of grad_output: torch.Size([1, 1, 1, 4])
grad_input: tensor([[[[ 0.5318,  0.1188,  0.1304, -0.1276],
          [ 0.2427,  0.0855,  0.0786, -0.9785],
          [ 0.2525,  0.0745,  0.5860, -0.1707],
          [ 0.1521,  0.5711,  0.1051, -0.19

In [63]:
def approxmax_pool2d(self, x_enc, kernel_size, stride=None, padding=0, ceil_mode=False, is_print=False):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
        
    x2_enc = x_enc.square()
    
    x_enc_avg2 = x_enc.mean().square()
    x_enc_var = x_enc.var()
    approx_r = 7 * x_enc_avg2 + 9 * x_enc_var 
    
    tau, i = 260, 0
    bound = 10             
    r = torch.tensor([0.1],device=self.device) 
    while(1):
        if approx_r.le(r).get_plain_text():
            break
        r *= 100
        tau /= 10
        i += 1
        if i >= bound:
            break
    x2_enc = tau**2 * x2_enc + 1
        
    # 套对应公式变形
    y = x_enc.mul(x2_enc)
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    x2_enc.share = torch.nn.functional.avg_pool2d(x2_enc.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    z = x2_enc.get_plain_text()
    z = torch.where(z > 1, torch.floor(z), z)

    z = y / z 
    return z, tau

In [64]:
def forward(
    ctx, input, kernel_size, padding=0, stride=None, dilation=1, ceil_mode=False
):
    # preprocess inputs:
    if stride is None:
        stride = kernel_size
    if isinstance(stride, (int, float)):
        stride = (stride, stride)
    if isinstance(padding, (int, float)):
        padding = (padding, padding)
    if isinstance(dilation, (int, float)):
        dilation = (dilation, dilation)
        
    # perform max pooling:
    output, tau = input.approxmax_pool2d(
        kernel_size,
        padding=padding,
        stride=stride,
        dilation=dilation,
        ceil_mode=ceil_mode
    ) # 为先前写好的最大池化优化最终版
    
    # store information for backward pass and return:
    ctx.save_multiple_for_backward(
        (input, tau, kernel_size, padding, stride, dilation, ceil_mode)
    )
    return output

In [62]:
def backward(ctx, grad_output):
    
    # get input, kernel_size, padding, stride, dilation, ceil_mode
    input, tau, kernel_size, padding, stride, dilation, ceil_mode = ctx.saved_tensors
    batches, channels, height, width = input.size()
    
    # unfold input to adapt to calculate input_weight
    input.share = torch.nn.functional.unfold(
        input.share.float(), kernel_size=kernel_size, dilation=dilation, padding=padding, stride=stride
    ) # (batch_size,channel_size*kernel_h*kernel_w,num_windows)
    input.share = input.share.long()
    unfolded_input = input.view(batches,channels,kernel_size**2,-1) # (batch_size,channel_size,kernel_h*kernel_w,num_windows)
    
    # calculate input_weight
    unfolded_input_2 = tau**2 * unfolded_input.square() + 1 # y = (ax)^2 + 1
    unfolded_input_sum = unfolded_input_2.sum(dim=2)
    mask_sum_max = torch.clamp(unfolded_input_sum.get_plain_text(),float('1e-2'), float('1e8')) # limit
    input_weight = unfolded_input_2 / mask_sum_max.unsqueeze(2)
    input_weight = input_weight.get_plain_text()
   
    # adjust grad_output to adapt to calculate grad_input
    grad_output = grad_output.view(batches,channels,1,-1)
    
    # calculate grad_input
    grad_input = grad_output*input_weight
    grad_input = grad_input.view(batches,channels*kernel_size**2,-1)
    grad_input.share = torch.nn.functional.fold(
        grad_input.share.float(), (height, width), kernel_size=kernel_size, dilation=dilation, padding=padding, stride=stride
    ) # fold grad_input to correspond input
    grad_input.share = grad_input.share.long()
    
    return grad_input

In [ ]:
def custom_unfold(input_tensor, kernel_size, stride=1, padding=0, dilation=1):
    # 获取输入张量的相关信息
    batch_size, num_channels, input_height, input_width = input_tensor.size()

    # 计算输出的高度和宽度
    output_height = (input_height + 2 * padding - dilation * (kernel_size[0] - 1) - 1) // stride + 1
    output_width = (input_width + 2 * padding - dilation * (kernel_size[1] - 1) - 1) // stride + 1

    # 使用unfold的思想，创建一个类似于unfold后的形状
    unfolded_shape = (batch_size, num_channels * kernel_size[0] * kernel_size[1], output_height, output_width)
    unfolded_tensor = torch.zeros(unfolded_shape, dtype=input_tensor.dtype, device=input_tensor.device)

    # 进行unfold操作
    for h in range(output_height):
        for w in range(output_width):
            h_start = h * stride - padding
            w_start = w * stride - padding
            h_end = h_start + dilation * (kernel_size[0] - 1) + 1
            w_end = w_start + dilation * (kernel_size[1] - 1) + 1

            unfolded_tensor[:, :, h, w] = input_tensor[:, :, h_start:h_end:dilation, w_start:w_end:dilation].reshape(batch_size, -1)

    return unfolded_tensor

In [ ]:
    @staticmethod
    def backward(ctx, grad_output):
        # get input, kernel_size, padding, stride, dilation, ceil_mode
        input, y2, kernel_size, padding, stride, dilation, ceil_mode = ctx.saved_tensors
        batches, channels, height, width = input.size()
        
        # 这里可以修改为直接由forward传递过来一个y2，然后再做shape变化，得到unfolded_input_2
        # unfold input to adapt to calculate input_weight
        y2 = y2 / 1e3
        print("y2-before:",y2.get_plain_text())
        y2.share = torch.nn.functional.unfold(
            y2.share.float(), kernel_size=kernel_size, dilation=dilation, padding=padding, stride=stride
        ) # (batch_size,channel_size*kernel_h*kernel_w,num_windows)
        
        y2.share = y2.share.long()
        y2 = y2 * 1e3
        print("y2-after:",y2.get_plain_text())
        unfolded_input_2 = y2.view(batches,channels,kernel_size**2,-1)
        
        unfolded_input_sum = unfolded_input_2.sum(dim=2)
        mask_sum_max = unfolded_input_sum.get_plain_text()
        input_weight = unfolded_input_2 / mask_sum_max.unsqueeze(2)
        input_weight = input_weight.get_plain_text()
   
        # adjust grad_output to adapt to calculate grad_input
        grad_output = grad_output.view(batches,channels,1,-1)
    
        # calculate grad_input
        print("grad_output:",grad_output.get_plain_text())
        print("input_weight:",input_weight)
        grad_input = grad_output*input_weight
        grad_input = grad_input.view(batches,channels*kernel_size**2,-1)
        grad_input.share = torch.nn.functional.fold(
            grad_input.share.float(), (height, width), kernel_size=kernel_size, dilation=dilation, padding=padding, stride=stride
        ) # fold grad_input to correspond input
        grad_input.share = grad_input.share.long()
        print("grad_input:",grad_input.get_plain_text())
    
        return grad_input

In [ ]:
def custom_fold(unfolded_input, output_size, kernel_size, stride, dilation=1, padding=0):
    # 获取展开后的数据形状
    batches, channels_times_k2, unfolded_height, unfolded_width = unfolded_input.size()

    # 计算输出的高度和宽度
    output_height, output_width = output_size

    # 计算输入的高度和宽度
    input_height = (output_height - 1) * stride + dilation * (kernel_size[0] - 1) + 1
    input_width = (output_width - 1) * stride + dilation * (kernel_size[1] - 1) + 1

    # 初始化一个空的张量来存储折叠后的数据
    folded_input = torch.zeros(batches, channels_times_k2 // (kernel_size[0] * kernel_size[1]), input_height, input_width)

    # 对每个滑动窗口进行折叠
    for i in range(unfolded_height):
        for j in range(unfolded_width):
            window = unfolded_input[:, :, i, j].contiguous()
            window = window.view(batches, -1, kernel_size[0], kernel_size[1])

            start_h = i * stride - padding
            start_w = j * stride - padding
            end_h = start_h + dilation * (kernel_size[0] - 1) + 1
            end_w = start_w + dilation * (kernel_size[1] - 1) + 1

            # 更新这里的折叠操作，使用直接赋值而不是 +=
            folded_input[:, :, start_h:end_h:dilation, start_w:end_w:dilation] = window

    return folded_input[:, :, :output_height, :output_width]